# Problem Statement

Unlocking Sales & Profit Insights for Sustainable Growth

Analyze XYZ Co.’s 2014–2018 sales data to uncover the key drivers of revenue and profitability across products, sales channels, and regions. Identify seasonal trends, performance gaps, and outliers, while comparing actual results against budget targets.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import pymysql 
import math
pd.set_option('display.max_columns', None)

In [2]:
sheets = pd.read_excel('Data/Regional Sales Dataset.xlsx', sheet_name = None)

# Assign dataframes to each sheet

df_sales = sheets['Sales Orders']
df_customers = sheets['Customers']
df_products = sheets['Products']
df_regions = sheets['Regions']
df_state_reg = sheets['State Regions']
df_budgets = sheets['2017 Budgets']

### Quick inspection


In [3]:
print(f"The shape of sales: {df_sales.shape}")
print(f"The shape of customers: {df_customers.shape}")
print(f"The shape of products: {df_products.shape}")
print(f"The shape of regions: {df_regions.shape}")
print(f"The shape of state regions: {df_state_reg.shape}")
print(f"The shape of budgets: {df_budgets.shape}")

The shape of sales: (64104, 12)
The shape of customers: (175, 2)
The shape of products: (30, 2)
The shape of regions: (994, 15)
The shape of state regions: (49, 3)
The shape of budgets: (30, 2)


In [4]:
df_state_reg.head(5)

,Column1,Column2,Column3
0,State Code,State,Region
1,AL,Alabama,South
2,AR,Arkansas,South
3,AZ,Arizona,West
4,CA,California,West


In [5]:
df_state_reg.columns = df_state_reg.iloc[0]      # set headers from row 0
df_state_reg = df_state_reg[1:].reset_index(drop=True)  # drop that row, reset index

In [6]:
df_budgets.head() 

,Product Name,2017 Budgets
0,Product 1,3016489.209
1,Product 2,3050087.565
2,Product 3,2642352.432
3,Product 4,2885560.824
4,Product 5,3925424.542


In [7]:
df_regions.head()

,id,name,county,state_code,state,type,latitude,longitude,area_code,population,households,median_income,land_area,water_area,time_zone
0,1,Auburn,Lee County,AL,Alabama,City,32.60986,-85.48078,334,62059,21767,38342,152375113,2646161,America/Chicago
1,2,Birmingham,Shelby County/Jefferson County,AL,Alabama,City,33.52744,-86.79905,205,212461,89972,31061,378353942,6591013,America/Chicago
2,3,Decatur,Limestone County/Morgan County,AL,Alabama,City,34.57332,-86.99214,256,55437,22294,41496,141006257,17594716,America/Chicago
3,4,Dothan,Dale County/Houston County/Henry County,AL,Alabama,City,31.23370,-85.40682,334,68567,25913,42426,232166237,835468,America/Chicago
4,5,Hoover,Shelby County/Jefferson County,AL,Alabama,City,33.37695,-86.80558,205,84848,32789,77146,122016784,2553332,America/Chicago


In [8]:
df_products.head()

,Index,Product Name
0,1,Product 1
1,2,Product 2
2,3,Product 3
3,4,Product 4
4,5,Product 5


In [9]:
df_customers.head()

,Customer Index,Customer Names
0,1,Geiss Company
1,2,Jaxbean Group
2,3,Ascend Ltd
3,4,Eire Corp
4,5,Blogtags Ltd


In [10]:
df_sales.head()

,OrderNumber,OrderDate,Customer Name Index,Channel,Currency Code,Warehouse Code,Delivery Region Index,Product Description Index,Order Quantity,Unit Price,Line Total,Total Unit Cost
0,SO - 000225,2014-01-01,126,Wholesale,USD,AXW291,364,27,6,2499.1,14994.6,1824.343
1,SO - 0003378,2014-01-01,96,Distributor,USD,AXW291,488,20,11,2351.7,25868.7,1269.918
2,SO - 0005126,2014-01-01,8,Wholesale,USD,AXW291,155,26,6,978.2,5869.2,684.740
3,SO - 0005614,2014-01-01,42,Export,USD,AXW291,473,7,7,2338.3,16368.1,1028.852
4,SO - 0005781,2014-01-01,73,Wholesale,USD,AXW291,256,8,8,2291.4,18331.2,1260.270


In [11]:
print(f"The info of sales: {df_sales.info()}")
print(f"The info of customers: {df_customers.info()}")
print(f"The info of products: {df_products.info()}")
print(f"The info of regions: {df_regions.info()}")
print(f"The info of state regions: {df_state_reg.info()}")
print(f"The info of budgets: {df_budgets.info()}")

<class 'pandas.DataFrame'>
RangeIndex: 64104 entries, 0 to 64103
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   OrderNumber                64104 non-null  str           
 1   OrderDate                  64104 non-null  datetime64[us]
 2   Customer Name Index        64104 non-null  int64         
 3   Channel                    64104 non-null  str           
 4   Currency Code              64104 non-null  str           
 5   Warehouse Code             64104 non-null  str           
 6   Delivery Region Index      64104 non-null  int64         
 7   Product Description Index  64104 non-null  int64         
 8   Order Quantity             64104 non-null  int64         
 9   Unit Price                 64104 non-null  float64       
 10  Line Total                 64104 non-null  float64       
 11  Total Unit Cost            64104 non-null  float64       
dtypes: datetime64[u

In [12]:
print(f"The description of sales: {df_sales.describe()}")
print(f"The description of customers: {df_customers.describe()}")
print(f"The description of products: {df_products.describe()}") 
print(f"The description of regions: {df_regions.describe()}")
print(f"The description of state regions: {df_state_reg.describe()}")
print(f"The description of budgets: {df_budgets.describe()}")

The description of sales:                         OrderDate  Customer Name Index  Delivery Region Index  \
count                       64104         64104.000000           64104.000000   
mean   2016-01-29 01:28:20.935979            87.480064             495.086609   
min           2014-01-01 00:00:00             1.000000               1.000000   
25%           2015-01-13 00:00:00            45.000000             247.000000   
50%           2016-01-27 00:00:00            87.000000             493.000000   
75%           2017-02-13 00:00:00           130.000000             742.000000   
max           2018-02-28 00:00:00           175.000000             994.000000   
std                           NaN            49.884946             285.645893   

       Product Description Index  Order Quantity    Unit Price    Line Total  \
count               64104.000000    64104.000000  64104.000000  64104.000000   
mean                   14.913141        8.441689   2284.380803  19280.682937   
min 

#### Cleaning Null value

In [13]:
print(f"The sum of null values in sales: {df_sales.isnull().sum()}")
print(f"The sum of null values in customers: {df_customers.isnull().sum()}")
print(f"The sum of null values in products: {df_products.isnull().sum()}")
print(f"The sum of null values in regions: {df_regions.isnull().sum()}")
print(f"The sum of null values in state regions: {df_state_reg.isnull().sum()}")
print(f"The sum of null values in budgets: {df_budgets.isnull().sum()}")

The sum of null values in sales: OrderNumber                  0
OrderDate                    0
Customer Name Index          0
Channel                      0
Currency Code                0
Warehouse Code               0
Delivery Region Index        0
Product Description Index    0
Order Quantity               0
Unit Price                   0
Line Total                   0
Total Unit Cost              0
dtype: int64
The sum of null values in customers: Customer Index    0
Customer Names    0
dtype: int64
The sum of null values in products: Index           0
Product Name    0
dtype: int64
The sum of null values in regions: id               0
name             0
county           0
state_code       0
state            0
type             0
latitude         0
longitude        0
area_code        0
population       0
households       0
median_income    0
land_area        0
water_area       0
time_zone        0
dtype: int64
The sum of null values in state regions: 0
State Code    0
State         0

In [14]:
dataframes = {
    "sales": df_sales,
    "customers": df_customers,
    "products": df_products,
    "regions": df_regions,
    "state regions": df_state_reg,
    "budgets": df_budgets
}

for name, df in dataframes.items():
    print(f"The duplicated rows in {name}: {df.duplicated().sum()}")

The duplicated rows in sales: 0
The duplicated rows in customers: 0
The duplicated rows in products: 0
The duplicated rows in regions: 0
The duplicated rows in state regions: 0
The duplicated rows in budgets: 0


In [15]:
df_sales.columns

Index(['OrderNumber', 'OrderDate', 'Customer Name Index', 'Channel',
       'Currency Code', 'Warehouse Code', 'Delivery Region Index',
       'Product Description Index', 'Order Quantity', 'Unit Price',
       'Line Total', 'Total Unit Cost'],
      dtype='str')

In [16]:
df_customers.columns

Index(['Customer Index', 'Customer Names'], dtype='str')

### Data Cleaning and Wrangling

In [17]:
# Merge sales and customers dataframes on the customer index

df = df_sales.merge(
    df_customers,
    how='left',
    left_on='Customer Name Index',
    right_on='Customer Index'
)

In [18]:
df.head()

,OrderNumber,OrderDate,Customer Name Index,Channel,Currency Code,Warehouse Code,Delivery Region Index,Product Description Index,Order Quantity,Unit Price,Line Total,Total Unit Cost,Customer Index,Customer Names
0,SO - 000225,2014-01-01,126,Wholesale,USD,AXW291,364,27,6,2499.1,14994.6,1824.343,126,Rhynoodle Ltd
1,SO - 0003378,2014-01-01,96,Distributor,USD,AXW291,488,20,11,2351.7,25868.7,1269.918,96,Thoughtmix Ltd
2,SO - 0005126,2014-01-01,8,Wholesale,USD,AXW291,155,26,6,978.2,5869.2,684.740,8,Amerisourc Corp
3,SO - 0005614,2014-01-01,42,Export,USD,AXW291,473,7,7,2338.3,16368.1,1028.852,42,Colgate-Pa Group
4,SO - 0005781,2014-01-01,73,Wholesale,USD,AXW291,256,8,8,2291.4,18331.2,1260.270,73,Deseret Group


In [19]:
df.drop(columns=['Customer Index'], inplace=True)

In [20]:
df.head()

,OrderNumber,OrderDate,Customer Name Index,Channel,Currency Code,Warehouse Code,Delivery Region Index,Product Description Index,Order Quantity,Unit Price,Line Total,Total Unit Cost,Customer Names
0,SO - 000225,2014-01-01,126,Wholesale,USD,AXW291,364,27,6,2499.1,14994.6,1824.343,Rhynoodle Ltd
1,SO - 0003378,2014-01-01,96,Distributor,USD,AXW291,488,20,11,2351.7,25868.7,1269.918,Thoughtmix Ltd
2,SO - 0005126,2014-01-01,8,Wholesale,USD,AXW291,155,26,6,978.2,5869.2,684.740,Amerisourc Corp
3,SO - 0005614,2014-01-01,42,Export,USD,AXW291,473,7,7,2338.3,16368.1,1028.852,Colgate-Pa Group
4,SO - 0005781,2014-01-01,73,Wholesale,USD,AXW291,256,8,8,2291.4,18331.2,1260.270,Deseret Group


In [21]:
# Merge the sales and products dataframes on the product index

df = df.merge(
    df_products,
    how='left',
    left_on='Product Description Index',
    right_on='Index'
)

In [22]:
df.head()

,OrderNumber,OrderDate,Customer Name Index,Channel,Currency Code,Warehouse Code,Delivery Region Index,Product Description Index,Order Quantity,Unit Price,Line Total,Total Unit Cost,Customer Names,Index,Product Name
0,SO - 000225,2014-01-01,126,Wholesale,USD,AXW291,364,27,6,2499.1,14994.6,1824.343,Rhynoodle Ltd,27,Product 27
1,SO - 0003378,2014-01-01,96,Distributor,USD,AXW291,488,20,11,2351.7,25868.7,1269.918,Thoughtmix Ltd,20,Product 20
2,SO - 0005126,2014-01-01,8,Wholesale,USD,AXW291,155,26,6,978.2,5869.2,684.740,Amerisourc Corp,26,Product 26
3,SO - 0005614,2014-01-01,42,Export,USD,AXW291,473,7,7,2338.3,16368.1,1028.852,Colgate-Pa Group,7,Product 7
4,SO - 0005781,2014-01-01,73,Wholesale,USD,AXW291,256,8,8,2291.4,18331.2,1260.270,Deseret Group,8,Product 8


In [23]:
df.drop(columns=['Index'], inplace=True)

In [24]:
df.head()

,OrderNumber,OrderDate,Customer Name Index,Channel,Currency Code,Warehouse Code,Delivery Region Index,Product Description Index,Order Quantity,Unit Price,Line Total,Total Unit Cost,Customer Names,Product Name
0,SO - 000225,2014-01-01,126,Wholesale,USD,AXW291,364,27,6,2499.1,14994.6,1824.343,Rhynoodle Ltd,Product 27
1,SO - 0003378,2014-01-01,96,Distributor,USD,AXW291,488,20,11,2351.7,25868.7,1269.918,Thoughtmix Ltd,Product 20
2,SO - 0005126,2014-01-01,8,Wholesale,USD,AXW291,155,26,6,978.2,5869.2,684.740,Amerisourc Corp,Product 26
3,SO - 0005614,2014-01-01,42,Export,USD,AXW291,473,7,7,2338.3,16368.1,1028.852,Colgate-Pa Group,Product 7
4,SO - 0005781,2014-01-01,73,Wholesale,USD,AXW291,256,8,8,2291.4,18331.2,1260.270,Deseret Group,Product 8


In [25]:
# Merge the sales and regions dataframes on the region index

df = df.merge(
    df_regions,
    how='left',
    left_on='Delivery Region Index',
    right_on='id'
)

In [26]:
df.head()

,OrderNumber,OrderDate,Customer Name Index,Channel,Currency Code,Warehouse Code,Delivery Region Index,Product Description Index,Order Quantity,Unit Price,Line Total,Total Unit Cost,Customer Names,Product Name,id,name,county,state_code,state,type,latitude,longitude,area_code,population,households,median_income,land_area,water_area,time_zone
0,SO - 000225,2014-01-01,126,Wholesale,USD,AXW291,364,27,6,2499.1,14994.6,1824.343,Rhynoodle Ltd,Product 27,364,Savannah,Chatham County,GA,Georgia,City,32.08354,-81.09983,912,145674,52798,36466,268318796,13908113,America/New York
1,SO - 0003378,2014-01-01,96,Distributor,USD,AXW291,488,20,11,2351.7,25868.7,1269.918,Thoughtmix Ltd,Product 20,488,Greenwood,Johnson County,IN,Indiana,City,39.61366,-86.10665,317,55586,20975,54176,72276415,1883,America/Indiana/Indianapolis
2,SO - 0005126,2014-01-01,8,Wholesale,USD,AXW291,155,26,6,978.2,5869.2,684.740,Amerisourc Corp,Product 26,155,Pleasanton,Alameda County,CA,California,City,37.66243,-121.87468,925,79510,26020,124759,62489257,386195,America/Los Angeles
3,SO - 0005614,2014-01-01,42,Export,USD,AXW291,473,7,7,2338.3,16368.1,1028.852,Colgate-Pa Group,Product 7,473,Bloomington,Monroe County,IN,Indiana,City,39.16533,-86.52639,812,84067,30232,30019,60221613,475857,America/Indiana/Indianapolis
4,SO - 0005781,2014-01-01,73,Wholesale,USD,AXW291,256,8,8,2291.4,18331.2,1260.270,Deseret Group,Product 8,256,Manchester,Hartford County,CT,Connecticut,Town,41.77524,-72.52443,959,58007,24141,63158,70972793,720300,America/New York


In [27]:
df.drop(columns=['id'], inplace=True)

In [28]:
df.columns

Index(['OrderNumber', 'OrderDate', 'Customer Name Index', 'Channel',
       'Currency Code', 'Warehouse Code', 'Delivery Region Index',
       'Product Description Index', 'Order Quantity', 'Unit Price',
       'Line Total', 'Total Unit Cost', 'Customer Names', 'Product Name',
       'name', 'county', 'state_code', 'state', 'type', 'latitude',
       'longitude', 'area_code', 'population', 'households', 'median_income',
       'land_area', 'water_area', 'time_zone'],
      dtype='str')

In [29]:
# Merge the sales and state regions dataframes on the state region index

df = df.merge(
    df_state_reg[['State Code', 'Region']],
    how='left',
    left_on='state_code',
    right_on='State Code'
)

In [30]:
df.head()

,OrderNumber,OrderDate,Customer Name Index,Channel,Currency Code,Warehouse Code,Delivery Region Index,Product Description Index,Order Quantity,Unit Price,Line Total,Total Unit Cost,Customer Names,Product Name,name,county,state_code,state,type,latitude,longitude,area_code,population,households,median_income,land_area,water_area,time_zone,State Code,Region
0,SO - 000225,2014-01-01,126,Wholesale,USD,AXW291,364,27,6,2499.1,14994.6,1824.343,Rhynoodle Ltd,Product 27,Savannah,Chatham County,GA,Georgia,City,32.08354,-81.09983,912,145674,52798,36466,268318796,13908113,America/New York,GA,South
1,SO - 0003378,2014-01-01,96,Distributor,USD,AXW291,488,20,11,2351.7,25868.7,1269.918,Thoughtmix Ltd,Product 20,Greenwood,Johnson County,IN,Indiana,City,39.61366,-86.10665,317,55586,20975,54176,72276415,1883,America/Indiana/Indianapolis,IN,Midwest
2,SO - 0005126,2014-01-01,8,Wholesale,USD,AXW291,155,26,6,978.2,5869.2,684.740,Amerisourc Corp,Product 26,Pleasanton,Alameda County,CA,California,City,37.66243,-121.87468,925,79510,26020,124759,62489257,386195,America/Los Angeles,CA,West
3,SO - 0005614,2014-01-01,42,Export,USD,AXW291,473,7,7,2338.3,16368.1,1028.852,Colgate-Pa Group,Product 7,Bloomington,Monroe County,IN,Indiana,City,39.16533,-86.52639,812,84067,30232,30019,60221613,475857,America/Indiana/Indianapolis,IN,Midwest
4,SO - 0005781,2014-01-01,73,Wholesale,USD,AXW291,256,8,8,2291.4,18331.2,1260.270,Deseret Group,Product 8,Manchester,Hartford County,CT,Connecticut,Town,41.77524,-72.52443,959,58007,24141,63158,70972793,720300,America/New York,CT,Northeast


In [31]:
df_budgets.columns

Index(['Product Name', '2017 Budgets'], dtype='str')

In [32]:
# Merge the sales and budgets dataframes on the year and region

df = df.merge(
    df_budgets,
    how='left',
    on='Product Name'
)

In [42]:
df.columns

Index(['OrderNumber', 'OrderDate', 'Customer Name Index', 'Channel',
       'Currency Code', 'Warehouse Code', 'Delivery Region Index',
       'Product Description Index', 'Order Quantity', 'Unit Price',
       'Line Total', 'Total Unit Cost', 'Customer Names', 'Product Name',
       'name', 'county', 'state_code', 'state', 'type', 'latitude',
       'longitude', 'area_code', 'population', 'households', 'median_income',
       'land_area', 'water_area', 'time_zone', 'Region', '2017 Budgets'],
      dtype='str')

In [33]:
df.drop(columns=['State Code'], inplace=True)

In [43]:
import re
import pandas as pd

def format_column(col):
    col = re.sub(r'(?<=[a-z0-9])(?=[A-Z])', '_', col)
    words = re.split(r'[\s_]+', col.strip())
    return '_'.join(w.capitalize() if not w.isupper() else w for w in words if w)

df.columns = [format_column(c) for c in df.columns]
print(df.columns.tolist())

['Order_Number', 'Order_Date', 'Customer_Name_Index', 'Channel', 'Currency_Code', 'Warehouse_Code', 'Delivery_Region_Index', 'Product_Description_Index', 'Order_Quantity', 'Unit_Price', 'Line_Total', 'Total_Unit_Cost', 'Customer_Names', 'Product_Name', 'Name', 'County', 'State_Code', 'State', 'Type', 'Latitude', 'Longitude', 'Area_Code', 'Population', 'Households', 'Median_Income', 'Land_Area', 'Water_Area', 'Time_Zone', 'Region', '2017_Budgets']


In [44]:
df.columns

Index(['Order_Number', 'Order_Date', 'Customer_Name_Index', 'Channel',
       'Currency_Code', 'Warehouse_Code', 'Delivery_Region_Index',
       'Product_Description_Index', 'Order_Quantity', 'Unit_Price',
       'Line_Total', 'Total_Unit_Cost', 'Customer_Names', 'Product_Name',
       'Name', 'County', 'State_Code', 'State', 'Type', 'Latitude',
       'Longitude', 'Area_Code', 'Population', 'Households', 'Median_Income',
       'Land_Area', 'Water_Area', 'Time_Zone', 'Region', '2017_Budgets'],
      dtype='str')

In [45]:
df.to_csv('Cleaned_Sales_Data.csv', index=False)

In [46]:
df.drop(columns=["Delivery_Region_Index"], inplace=True)

In [47]:
df.columns

Index(['Order_Number', 'Order_Date', 'Customer_Name_Index', 'Channel',
       'Currency_Code', 'Warehouse_Code', 'Product_Description_Index',
       'Order_Quantity', 'Unit_Price', 'Line_Total', 'Total_Unit_Cost',
       'Customer_Names', 'Product_Name', 'Name', 'County', 'State_Code',
       'State', 'Type', 'Latitude', 'Longitude', 'Area_Code', 'Population',
       'Households', 'Median_Income', 'Land_Area', 'Water_Area', 'Time_Zone',
       'Region', '2017_Budgets'],
      dtype='str')

In [48]:
cols_to_keep = [
    'Order_Number',
    'Order_Date',
    'Customer_Names',
    'Channel',
    'Product_Name',
    'Order_Quantity',
    'Unit_Price',
    'Line_Total',
    'Total_Unit_Cost',
    'State_Code',
    'County',
    'State',
    'Region',
    'Latitude',
    'Longitude',
    '2017_Budgets'
]

In [49]:
final_df = df[cols_to_keep]

In [50]:
final_df.head()

,Order_Number,Order_Date,Customer_Names,Channel,Product_Name,Order_Quantity,Unit_Price,Line_Total,Total_Unit_Cost,State_Code,County,State,Region,Latitude,Longitude,2017_Budgets
0,SO - 000225,2014-01-01,Rhynoodle Ltd,Wholesale,Product 27,6,2499.1,14994.6,1824.343,GA,Chatham County,Georgia,South,32.08354,-81.09983,964940.231
1,SO - 0003378,2014-01-01,Thoughtmix Ltd,Distributor,Product 20,11,2351.7,25868.7,1269.918,IN,Johnson County,Indiana,Midwest,39.61366,-86.10665,2067108.120
2,SO - 0005126,2014-01-01,Amerisourc Corp,Wholesale,Product 26,6,978.2,5869.2,684.740,CA,Alameda County,California,West,37.66243,-121.87468,5685138.270
3,SO - 0005614,2014-01-01,Colgate-Pa Group,Export,Product 7,7,2338.3,16368.1,1028.852,IN,Monroe County,Indiana,Midwest,39.16533,-86.52639,889737.555
4,SO - 0005781,2014-01-01,Deseret Group,Wholesale,Product 8,8,2291.4,18331.2,1260.270,CT,Hartford County,Connecticut,Northeast,41.77524,-72.52443,1085037.329


In [55]:
dff = final_df.copy()

In [56]:
dff.columns

Index(['Order_Number', 'Order_Date', 'Customer_Names', 'Channel',
       'Product_Name', 'Order_Quantity', 'Unit_Price', 'Line_Total',
       'Total_Unit_Cost', 'State_Code', 'County', 'State', 'Region',
       'Latitude', 'Longitude', '2017_Budgets'],
      dtype='str')

In [57]:
dff.loc[dff["Order_Date"].dt.year != 2017, "2017_Budgets"] = pd.NA

In [61]:
dff.tail()

,Order_Number,Order_Date,Customer_Names,Channel,Product_Name,Order_Quantity,Unit_Price,Line_Total,Total_Unit_Cost,State_Code,County,State,Region,Latitude,Longitude,2017_Budgets
64099,SO - 0007573,2018-02-28,Dazzlesphe Corp,Wholesale,Product 26,12,1815.7,21788.4,980.478,PA,Bucks County,Pennsylvania,Northeast,40.15511,-74.82877,NaN
64100,SO - 0007706,2018-02-28,Yombu Corp,Export,Product 21,6,864.3,5185.8,579.081,IL,Cook County,Illinois,Midwest,42.11030,-88.03424,NaN
64101,SO - 0007718,2018-02-28,Bath Group,Distributor,Product 13,11,3953.0,43483.0,2648.510,FL,Broward County,Florida,South,26.24453,-80.20644,NaN
64102,SO - 0008084,2018-02-28,Linklinks Ltd,Distributor,Product 20,7,3959.7,27717.9,2930.178,NY,Erie County,New York,Northeast,42.91002,-78.74182,NaN
64103,SO - 0008654,2018-02-28,SAFEWAY Ltd,Distributor,Product 15,8,998.3,7986.4,848.555,OR,Washington County,Oregon,West,45.48706,-122.80371,NaN


In [62]:
dff.to_csv("final_df.csv")

In [63]:
dff.info()

<class 'pandas.DataFrame'>
RangeIndex: 64104 entries, 0 to 64103
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Order_Number     64104 non-null  str           
 1   Order_Date       64104 non-null  datetime64[us]
 2   Customer_Names   64104 non-null  str           
 3   Channel          64104 non-null  str           
 4   Product_Name     64104 non-null  str           
 5   Order_Quantity   64104 non-null  int64         
 6   Unit_Price       64104 non-null  float64       
 7   Line_Total       64104 non-null  float64       
 8   Total_Unit_Cost  64104 non-null  float64       
 9   State_Code       64104 non-null  str           
 10  County           64104 non-null  str           
 11  State            64104 non-null  str           
 12  Region           64104 non-null  str           
 13  Latitude         64104 non-null  float64       
 14  Longitude        64104 non-null  float64       
 